# BeanGuard — Bean Leaf Disease Classifier — Model Development Notebook

**Problem.** Beans are a staple crop in Rwanda and across East Africa, and leaf diseases such as *angular leaf spot* and
*bean rust* can destroy a harvest if they are not spotted early. Agronomists are few and fields are many. This project
builds an image classifier that recognises **healthy, angular leaf spot and bean rust** leaves from a phone photo, and —
crucially — keeps improving as extension officers upload new labelled photos (see the retraining pipeline in
`src/retrain.py`).

**Dataset.** iBean (Makerere AI Lab, Uganda): ~1,300 labelled bean-leaf photos taken in real fields.

**Where this notebook sits in the lifecycle**

`Data acquisition → Preprocessing → Model creation → Evaluation (this notebook)` → `FastAPI` → `Dashboard` →
`Retraining` → `Docker / cloud` → `Locust load test & monitoring`

**How to run**
* *Local:* from the repo root run `jupyter lab`, open `notebook/rwanda_heritage_classifier.ipynb`.
* *Google Colab (GPU recommended):* upload the project zip, unzip, `%cd` into it, set the runtime to GPU, run all.

**Rubric map:** preprocessing steps (§2–3) · optimisation techniques (§5, §6) · ≥ 4 evaluation metrics (§7) ·
test/prediction functions (§8) · saved model files (§9).

## 0. Setup

In [ ]:
import json, random, shutil, hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps

import tensorflow as tf
from tensorflow.keras import layers, regularizers
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             log_loss, precision_score, recall_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Works whether the notebook is opened from the repo root or from notebook/
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
TRAIN_DIR, TEST_DIR = ROOT / "data" / "train", ROOT / "data" / "test"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("Repo root:", ROOT)

## 1. Data acquisition

**Classes** (folder name = label): `healthy`, `angular_leaf_spot`, `bean_rust`.

**Source:** the public **iBean dataset** (Makerere AI Lab, Uganda) — field photos of bean leaves, downloaded with
`python scripts/download_beans.py` (record the source and licence in `data/SOURCES.md`).

**Pipeline.** Raw images are kept in `raw_images/<class>/`. `scripts/split_dataset.py` then
1. validates every file and drops corrupt images,
2. converts to RGB JPEG (max side 512 px) and removes exact duplicates (MD5),
3. splits **per class** into `train` (75 %), `test` (15 %, *never* used for training or retraining) and
   `new_data_demo` (10 %, held back to demonstrate the retraining upload).

```bash
python scripts/download_beans.py
python scripts/split_dataset.py raw_images --out data
```

In [ ]:
# Verify the data layout and show class counts for both splits
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
def count(d): return {c: sum(1 for f in (d / c).glob("*") if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
                      for c in CLASS_NAMES}
counts = pd.DataFrame({"train": count(TRAIN_DIR), "test": count(TEST_DIR)})
counts["total"] = counts.sum(axis=1)
display(counts)
print("Classes:", CLASS_NAMES)

## 2. Exploratory data analysis — *what story does the data tell?*

In [ ]:
ax = counts[["train", "test"]].plot(kind="bar", stacked=True, figsize=(8, 4), color=["#2b6cb0", "#ed8936"])
ax.set_title("Images per class"); ax.set_ylabel("images"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()
imbalance = counts["train"].max() / counts["train"].min()
print(f"Imbalance ratio (largest / smallest class): {imbalance:.2f}x")

In [ ]:
# A sample grid: 4 random images per class
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(10, 2.4 * len(CLASS_NAMES)))
for r, cls in enumerate(CLASS_NAMES):
    files = random.sample(sorted((TRAIN_DIR / cls).glob("*")), k=min(4, len(list((TRAIN_DIR / cls).glob("*")))))
    for c in range(4):
        axes[r, c].axis("off")
        if c < len(files):
            axes[r, c].imshow(Image.open(files[c]).convert("RGB"))
        if c == 0:
            axes[r, c].set_title(cls, loc="left", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Per-image features: size, aspect ratio, brightness, contrast, colour saturation
def image_features(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im)
        w, h = im.size
        small = im.convert("RGB").resize((96, 96))
        gray = np.asarray(small.convert("L"), dtype="float32")
        sat = np.asarray(small.convert("HSV"), dtype="float32")[..., 1]
    return dict(width=w, height=h, aspect_ratio=w / h, brightness=gray.mean(), contrast=gray.std(), saturation=sat.mean())

rows = []
for cls in CLASS_NAMES:
    files = sorted((TRAIN_DIR / cls).glob("*"))
    for f in files[:: max(len(files) // 100, 1)][:100]:
        rows.append({"class": cls, **image_features(f)})
feat = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["brightness", "saturation", "aspect_ratio"]):
    sns.boxplot(data=feat, x="class", y=col, ax=ax); ax.set_title(col.replace("_", " ").title()); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
display(feat.groupby("class")[["brightness", "contrast", "saturation", "aspect_ratio"]].mean().round(2))

### Interpretation of three features  *(fill in after running — use the numbers above)*

1. **Class balance** — *e.g.* "`healthy` has N images vs M for `bean_rust`; I apply balanced class weights and judge
   the model by per-class recall and macro-F1, not accuracy alone."
2. **Colour saturation** — *e.g.* "Healthy leaves are uniformly green (low spread of saturation) while rust-affected
   leaves show rust-coloured pustules, so saturation separates them; angular leaf spot is subtler."
3. **Brightness / contrast** — *e.g.* "Photos taken in bright sun vs shade differ in mean brightness (X vs Y);
   random contrast augmentation stops the model relying on lighting."
4. **Aspect ratio** — *e.g.* "Most photos are square/landscape, so resizing to 224×224 distorts little."

## 3. Preprocessing

| Step | What | Why |
|---|---|---|
| Validation | Corrupt/duplicate files removed (`split_dataset.py`) | Prevent crashes & leakage |
| EXIF orientation fix | Phone photos rotated upright | Avoid sideways inputs |
| Resize | 224 × 224 RGB | MobileNetV2 input size |
| Scaling | Pixels → [-1, 1] **inside the model** (`Rescaling` layer) | Training & serving cannot drift apart |
| Split | 80 / 20 train / validation of `data/train` (seeded); `data/test` untouched | Honest final evaluation |
| Class weights | Balanced weights | Handles class imbalance |
| Augmentation | Flip, rotation, zoom, contrast (training only) | Regularisation for small data |
| Pipeline | `tf.data` with `prefetch` | Fast, GPU-friendly input |

In [ ]:
def make_ds(directory, validation_split=None, subset=None, shuffle=True):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory, labels="inferred", label_mode="int", class_names=CLASS_NAMES, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED, validation_split=validation_split, subset=subset)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(TRAIN_DIR, 0.2, "training")
val_ds   = make_ds(TRAIN_DIR, 0.2, "validation")
test_ds  = make_ds(TEST_DIR, shuffle=False)

n_total = counts["train"].sum()
CLASS_WEIGHT = {i: n_total / (len(CLASS_NAMES) * max(counts.loc[c, "train"], 1)) for i, c in enumerate(CLASS_NAMES)}
print({CLASS_NAMES[i]: round(w, 2) for i, w in CLASS_WEIGHT.items()})

augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"), layers.RandomRotation(0.08),
    layers.RandomZoom(0.15), layers.RandomContrast(0.15)], name="augmentation")

# Visual check of the augmentation
imgs, _ = next(iter(train_ds.take(1)))
plt.figure(figsize=(10, 5))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmentation(imgs[:1], training=True)[0].numpy().astype("uint8")); plt.axis("off")
plt.suptitle("Augmented versions of one training image"); plt.show()

## 4. Baseline — a vanilla CNN trained from scratch
A deliberately simple model, to **prove the value of the optimisation techniques** used afterwards.

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="val"); ax[0].set_title(f"{title} - accuracy"); ax[0].legend()
    ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="val"); ax[1].set_title(f"{title} - loss"); ax[1].legend()
    plt.tight_layout(); plt.show()

baseline = tf.keras.Sequential([
    layers.Input((*IMG_SIZE, 3)), layers.Rescaling(1 / 255),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dense(128, activation="relu"), layers.Dense(len(CLASS_NAMES), activation="softmax"),
], name="baseline_cnn")
baseline.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_base = baseline.fit(train_ds, validation_data=val_ds, epochs=10, verbose=2)
plot_history(h_base.history, "Baseline CNN")

## 5. Optimised model — MobileNetV2 transfer learning

**Optimisation techniques used:** ImageNet pre-trained backbone · data augmentation · Dropout (0.3) · L2 weight decay ·
Adam optimiser with a **two-phase schedule** (frozen backbone @ 1e-3, then fine-tuning the top 40 layers @ 1e-5) ·
EarlyStopping · ReduceLROnPlateau · balanced class weights.

In [ ]:
def build_model(num_classes, dropout=0.3, l2=1e-4):
    backbone = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
    backbone.trainable = False
    inputs = layers.Input((*IMG_SIZE, 3), name="image")
    x = augmentation(inputs)                                   # only active in training
    x = layers.Rescaling(1 / 127.5, offset=-1, name="rescale")(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(num_classes, activation="softmax", kernel_regularizer=regularizers.l2(l2), name="probs")(x)
    return tf.keras.Model(inputs, out, name="heritage_classifier")

def callbacks(patience=4):
    return [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7)]

def compile_model(m, lr):
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def unfreeze_top(m, n=40):
    backbone = next(l for l in m.layers if isinstance(l, tf.keras.Model))
    backbone.trainable = True
    for l in backbone.layers[:-n]: l.trainable = False
    for l in backbone.layers:
        if isinstance(l, layers.BatchNormalization): l.trainable = False
    return m

### 5.1 Small hyper-parameter search (frozen backbone, 4 epochs each)

In [ ]:
results = []
for dropout in [0.2, 0.4]:
    for lr in [1e-3, 3e-4]:
        m = compile_model(build_model(len(CLASS_NAMES), dropout=dropout), lr)
        h = m.fit(train_ds, validation_data=val_ds, epochs=4, class_weight=CLASS_WEIGHT, verbose=0)
        results.append({"dropout": dropout, "learning_rate": lr, "val_accuracy": max(h.history["val_accuracy"]),
                        "val_loss": min(h.history["val_loss"])})
        print(results[-1])
hp = pd.DataFrame(results).sort_values("val_loss"); display(hp)
BEST_DROPOUT = float(hp.iloc[0]["dropout"]); BEST_LR = float(hp.iloc[0]["learning_rate"])
print("Selected:", BEST_DROPOUT, BEST_LR)

### 5.2 Final training — phase 1 (new head) then phase 2 (fine-tuning)

In [ ]:
model = compile_model(build_model(len(CLASS_NAMES), dropout=BEST_DROPOUT), BEST_LR)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=CLASS_WEIGHT, callbacks=callbacks(), verbose=2)

unfreeze_top(model, 40)
compile_model(model, 1e-5)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=CLASS_WEIGHT, callbacks=callbacks(), verbose=2)

history = {k: [float(x) for x in h1.history[k] + h2.history[k]] for k in h1.history}
plot_history(history, "MobileNetV2 (head + fine-tune)")

## 6. Evaluation on the held-out test set
Metrics: **accuracy, loss (log-loss), precision, recall, F1** (macro & weighted), per-class report and confusion matrix.

In [ ]:
def evaluate(m, ds):
    y_true, y_prob = [], []
    for x, y in ds:
        y_prob.append(m.predict(x, verbose=0)); y_true.append(y.numpy())
    y_true, y_prob = np.concatenate(y_true), np.concatenate(y_prob)
    y_pred = y_prob.argmax(1); labels = list(range(len(CLASS_NAMES)))
    return {
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "loss": float(log_loss(y_true, y_prob, labels=labels)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "per_class": classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
        "class_names": CLASS_NAMES,
    }, y_true, y_pred, y_prob

test_metrics, y_true, y_pred, y_prob = evaluate(model, test_ds)
base_metrics, *_ = evaluate(baseline, test_ds)

cmp = pd.DataFrame({"Baseline CNN": base_metrics, "MobileNetV2 (optimised)": test_metrics}).loc[
    ["accuracy", "loss", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]].astype(float).round(4)
display(cmp)

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
cm = np.array(test_metrics["confusion_matrix"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix (test set)"); plt.show()

**Discussion** *(write after running)*: which classes are confused and why (link back to the EDA features),
how much the optimised model improves on the baseline, and what the confidence distribution says about the
`other` class. Limitations: dataset size, photo-source bias, lighting.

## 7. Prediction & test functions

In [ ]:
def predict_image(path, top_k=3):
    """Single-image prediction - same logic as src/prediction.py."""
    img = Image.open(path); img = ImageOps.exif_transpose(img).convert("RGB").resize(IMG_SIZE)
    probs = model(np.expand_dims(np.asarray(img, dtype="float32"), 0), training=False).numpy()[0]
    order = probs.argsort()[::-1][:top_k]
    return [(CLASS_NAMES[i], float(probs[i])) for i in order]

paths = [(p, c) for c in CLASS_NAMES for p in sorted((TEST_DIR / c).glob("*"))]
sample = random.sample(paths, k=min(12, len(paths)))
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, (p, truth) in zip(axes.ravel(), sample):
    label, conf = predict_image(p)[0]
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
    ax.set_title(f"{label} ({conf:.0%})\ntrue: {truth}", color="green" if label == truth else "red", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Error analysis: the most confident mistakes
files = [p for p, _ in paths]  # same order as test_ds (shuffle=False, class-sorted)
wrong = np.where(y_true != y_pred)[0]
worst = sorted(wrong, key=lambda i: -y_prob[i].max())[:6]
print(f"{len(wrong)} / {len(y_true)} test images misclassified")
if len(worst):
    fig, axes = plt.subplots(1, len(worst), figsize=(3 * len(worst), 3.4))
    for ax, i in zip(np.atleast_1d(axes), worst):
        ax.imshow(Image.open(files[i]).convert("RGB")); ax.axis("off")
        ax.set_title(f"pred {CLASS_NAMES[y_pred[i]]}\ntrue {CLASS_NAMES[y_true[i]]}", fontsize=9, color="red")
    plt.show()
plt.hist(y_prob.max(1), bins=20); plt.title("Confidence on the test set"); plt.xlabel("max softmax probability"); plt.show()

## 8. Save the model artefacts
Everything the API, dashboard and retraining pipeline need: the model (`.keras` served by the API + `.h5` copy),
the class list and a metrics file with the same schema `src/model.py` writes.

In [ ]:
model.save(MODEL_DIR / "heritage_mobilenetv2.keras")
try:
    model.save(MODEL_DIR / "heritage_mobilenetv2.h5")
except Exception as e:
    print("h5 export skipped:", e)
(MODEL_DIR / "class_names.json").write_text(json.dumps(CLASS_NAMES))
(MODEL_DIR / "metrics.json").write_text(json.dumps({
    "version": 1, "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "trigger": "initial",
    "n_train": int(counts["train"].sum()), "test": test_metrics, "history": history}))
print(sorted(p.name for p in MODEL_DIR.iterdir()))

# On Google Colab: download the artefacts, then copy them into the repo's models/ folder
# from google.colab import files
# !cd models && zip -r ../models.zip . && cd .. && python -c "from google.colab import files; files.download('models.zip')"

## 9. What happens next
* `src/prediction.py` + `api/main.py` serve this model (FastAPI) · `ui/app.py` is the dashboard.
* New labelled images are uploaded in the dashboard → `src/retrain.py` **fine-tunes this saved model**, compares it with the
  current model on the same test set, and promotes it only if macro-F1 does not drop.
* `docker-compose.yml` + `locust/` deploy and load-test the system (see README).